In [1]:
from obtain_allocation import *
from study_results_per_line import *
from generate_gaussian import *
import os
import pandas as pd
import numpy as np
import matplotlib as plt

# ANALISIS toA

In [2]:
def generate_gaussian_per_slot_all_days(dataframeinput: pd.DataFrame, param_to_gauss: str):
    """
    Analyzes 'ALL' workdays. If initial normality fails, it filters outliers
    and replaces the results with the cleaned dataset statistics.
    """

    dataframe = dataframeinput[(dataframeinput["Occupation"] <= 80) & (dataframeinput["Occupation"] >= 0)]
    dataframe = dataframe[(dataframe["DayType"] == "LA") & (dataframe["FreeDay"] == 0)]

    if dataframe.empty:
        return pd.DataFrame()

    stopnum = dataframe["Stop"].iloc[0]
    timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]

    columns = ["day", "Stop"]
    for time in timeslot:
        columns.extend([
            f"Mean_{time}",
            f"std_{time}",
            f"IsNormal_{time}",
            f"outliers_{time}",
            f"instances_{time}"
        ])

    row_data = ["ALL", stopnum]

    for time in timeslot:
        data_per_timeslot = dataframe[dataframe["TimeSlot"] == time].copy()

        gauss_hypothesis, instances_involved = is_normal_modified(
            data_per_timeslot, "ALL", time, param_to_gauss, False
        )

        outliers_found = []

        if not isinstance(instances_involved, list) and not instances_involved.empty:
            current_values = instances_involved[param_to_gauss].tolist()
            outliers_found = estimate_outliers(current_values, 10, False)

            if not gauss_hypothesis and len(outliers_found) > 0:
                # Filter out the outliers
                data_cleaned = data_per_timeslot[~data_per_timeslot[param_to_gauss].isin(outliers_found)]

                new_hypothesis, new_instances = is_normal_modified(
                    data_cleaned, "ALL", time, param_to_gauss, False
                )

                if not isinstance(new_instances, list) and not new_instances.empty:
                    gauss_hypothesis = new_hypothesis
                    instances_involved = new_instances
                else:
                    # If removing outliers destroyed the dataset, mark as invalid
                    instances_involved = []

        # --- Final Stats Calculation ---
        if not isinstance(instances_involved, list) and not instances_involved.empty:
            final_values = instances_involved[param_to_gauss].tolist()

            mean_val = np.mean(final_values)
            std_val = sqrt(central_moment(final_values, 2))
            instances_list = instances_involved["Occupation"].tolist()

            row_data.extend([mean_val, std_val, gauss_hypothesis, outliers_found, instances_list])
        else:
            row_data.extend([0, 0, "NO DATA", 0, 0])

    new_dataframe = pd.DataFrame([row_data], columns=columns)
    return new_dataframe

In [3]:
files_stops = files = os.listdir("../data/E01_toA/NEW")


#Creamos plantilla para el dataframe FINAL

daytype = ["L", "M", "X", "J", "V"]
timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


#Inittialy column day to specify L, M, X, J, V
columns = ["day"]
columns.append("Stop")

#For each timeslot t add columns Mean_t, std_t and IsNormal_t
for time in timeslot:
    strvar = f"Mean_{time}"
    columns.append(strvar)
    strvar = f"std_{time}"
    columns.append(strvar)
    strvar = f"IsNormal_{time}"
    columns.append(strvar)
    strvar = f"outliers_{time}"
    columns.append(strvar)
    strvar = f"instances_{time}"
    columns.append(strvar)

#generate df
final_dataframe = pd.DataFrame(columns=columns)

for file in files_stops:
    if file != "RESULTS" and file != "E01_toA.zip":
        dataframe = pd.read_csv("../data/E01_toA/NEW/" + file, encoding="utf-8", header=0)
        dataframe = clean_dataframe(dataframe, 2)
        new_dataframe = generate_gaussian_per_slot_all_days(dataframe, "Occupation")
        final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)

with open(f"../data/E01_toA/NEW/RESULTS/FINAL_results_NORMAL.csv", "w", encoding="utf-8") as f:
        final_dataframe.to_csv(f, index=False)

/tmp/ipykernel_132481/2265873351.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)


# toB

In [4]:
files_stops = files = os.listdir("../data/E01_toB")


#Creamos plantilla para el dataframe FINAL

daytype = ["L", "M", "X", "J", "V"]
timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


#Inittialy column day to specify L, M, X, J, V
columns = ["day"]
columns.append("Stop")

#For each timeslot t add columns Mean_t, std_t and IsNormal_t
for time in timeslot:
    strvar = f"Mean_{time}"
    columns.append(strvar)
    strvar = f"std_{time}"
    columns.append(strvar)
    strvar = f"IsNormal_{time}"
    columns.append(strvar)
    strvar = f"outliers_{time}"
    columns.append(strvar)
    strvar = f"instances_{time}"
    columns.append(strvar)

#generate df
final_dataframe = pd.DataFrame(columns=columns)

for file in files_stops:
    if file != "RESULTS" and file != "E01_toB.zip":
        dataframe = pd.read_csv("../data/E01_toB/" + file, encoding="utf-8", header=0)
        dataframe = clean_dataframe(dataframe, 1)
        new_dataframe = generate_gaussian_per_slot_all_days(dataframe, "Occupation")
        final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)

with open(f"../data/E01_toB/RESULTS/FINAL_results_NORMAL.csv", "w", encoding="utf-8") as f:
        final_dataframe.to_csv(f, index=False)

/tmp/ipykernel_132481/535983161.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)
